# Pré-processamento

**Entrega: 07/11/2026 | Valor: 4,00 pontos.**

As partes 1, 2 e 4 estão implementadas. As partes 3 e 5 permanecem como roteiro de trabalho. Substituir Membro 1–5 pelos nomes do grupo do CANVAS.


## Regras comuns às cinco tarefas

- Usar os CSVs oficiais em `data/`: `treino.csv`, `teste_sem_rotulo.csv` e `modelo_de_submissao.csv`.
- O alvo é `class`, com valores `normal` e `anomaly`; não remapear para cinco famílias de ataques.
- Não substituir, reembaralhar ou recriar a separação oficial de treino/teste. Validação interna é permitida **somente dentro do treino**.
- Ajustar imputação, codificação, normalização, balanceamento e redução apenas no treino de cada dobra.
- `id` identifica o teste e não é atributo preditivo. O teste não tem gabarito disponível à equipe.
- Investigar a associação de **cada atributo** com a classe antes de treinar modelos. Correlação alta não prova vazamento sozinha.
- Examinar também as colunas adicionadas, inclusive `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min`; justificar decisões pela origem e disponibilidade da informação.
- Não reutilizar conclusões ou números da base antiga. Não procurar/reconstruir rótulos do teste público.
- Os módulos em `src/` são a estrutura legada e precisam ser adaptados pelos responsáveis; este roteiro não os executa.
- Cada membro deve produzir código comentado, resultados e uma conclusão autoral em sua seção, além de revisar a integração.


## Membro 1 — Ausentes, redundâncias e inconsistências

**Tarefa:** Implementar a carga comum e as estratégias indicadas pela etapa 1. Comparar tratamento de ausentes, normalização de categorias e manutenção/remoção de redundâncias com uma referência. Definir tratamento de duplicatas/conflitos sem contaminar validação e sem alterar o teste oficial.

**Entregável:** Transformações reutilizáveis e tabela de estratégias, evidências e justificativas. Nenhuma mediana ou decisão estatística aprendida fora das dobras.


### 1.1 Carga comum

Funções `carregar_treino()` e `carregar_teste()` para todos os membros. Os CSVs oficiais não são alterados, o alvo vira `y` (1 = `anomaly`), e `id` nunca entra como atributo.
Os sinais operacionais (`score_ids_legado`, `tipo_alerta_soc`, `tempo_resposta_analista_min`) ficam **fora** dos preditores por padrão, porque a etapa 1 indicou risco de vazamento (o tempo do analista existe sobretudo após a triagem). A decisão final é do Membro 5; basta passar `incluir_operacionais=True` para reavaliar.

In [1]:
# Membro 1 — 1.1 Carga comum (reutilizável pelos demais membros)
from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

SEMENTE = 42
N_DOBRAS = 5

project_root = Path.cwd().resolve()
if not (project_root / "data" / "treino.csv").exists():
    project_root = project_root.parent
if not (project_root / "data" / "treino.csv").exists():
    raise FileNotFoundError("Execute na raiz do projeto ou em notebooks/.")
DIR_DADOS = project_root / "data"

COL_ROTULO = "class"
COL_ID = "id"
CLASSES = ["normal", "anomaly"]          # anomaly = classe positiva (1)
COLS_CATEGORICAS = ["protocol_type", "service", "flag"]
# Sinais operacionais: ficam fora dos preditores até a conclusão do Membro 5 (etapa 1, seção 1.7).
SINAIS_OPERACIONAIS = ["score_ids_legado", "tipo_alerta_soc", "tempo_resposta_analista_min"]
# Colunas derivadas -> coluna de origem e relação esperada (verificadas na seção 1.2).
COLS_DERIVADAS = {
    "src_bytes_kb": ("src_bytes", "src_bytes / 1024"),
    "dst_bytes_kb": ("dst_bytes", "dst_bytes / 1024"),
    "total_bytes": ("src_bytes + dst_bytes", "src_bytes + dst_bytes"),
    "same_srv_pct": ("same_srv_rate", "same_srv_rate * 100"),
    "protocolo_cod": ("protocol_type", "código 1:1 de protocol_type"),
}
# Limites de domínio da documentação do KDD (conhecimento prévio, não aprendidos dos dados).
LIMITES_DOMINIO = {
    **{c: (0, 1) for c in ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
                           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
                           "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
                           "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
                           "dst_host_serror_rate", "dst_host_srv_serror_rate",
                           "dst_host_rerror_rate", "dst_host_srv_rerror_rate"]},
    "same_srv_pct": (0, 100),
    "count": (0, 511), "srv_count": (0, 511),
    "dst_host_count": (0, 255), "dst_host_srv_count": (0, 255),
    "duration": (0, np.inf), "src_bytes": (0, np.inf), "dst_bytes": (0, np.inf),
    "total_bytes": (0, np.inf), "src_bytes_kb": (0, np.inf), "dst_bytes_kb": (0, np.inf),
}


def carregar_treino(incluir_operacionais: bool = False):
    """Lê treino.csv sem alterá-lo. Retorna (X, y) com y = 1 para anomaly."""
    df = pd.read_csv(DIR_DADOS / "treino.csv")
    assert set(df[COL_ROTULO]) == set(CLASSES) and df[COL_ROTULO].notna().all()
    assert COL_ID not in df.columns, "id não é atributo preditivo."
    descartar = [COL_ROTULO] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    X = df.drop(columns=descartar)
    y = (df[COL_ROTULO] == "anomaly").astype(int).rename(COL_ROTULO)
    return X, y


def carregar_teste(incluir_operacionais: bool = False):
    """Lê teste_sem_rotulo.csv somente para checar esquema/aplicar o pipeline já ajustado.

    Nenhuma estatística é calculada com o teste; ordem e IDs são preservados.
    """
    df = pd.read_csv(DIR_DADOS / "teste_sem_rotulo.csv")
    ids = df[COL_ID].copy()
    descartar = [COL_ID] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    return ids, df.drop(columns=descartar)


X_treino, y_treino = carregar_treino()
print(f"Treino: {X_treino.shape[0]:,} linhas x {X_treino.shape[1]} atributos candidatos "
      f"(sinais operacionais excluídos: {SINAIS_OPERACIONAIS})")
print("Distribuição do alvo:", y_treino.map({0: "normal", 1: "anomaly"}).value_counts().to_dict())

Treino: 69,833 linhas x 46 atributos candidatos (sinais operacionais excluídos: ['score_ids_legado', 'tipo_alerta_soc', 'tempo_resposta_analista_min'])
Distribuição do alvo: {'normal': 65754, 'anomaly': 4079}


### 1.2 Evidências de redundância e inconsistência

Antes de comparar estratégias, confirmamos com igualdades exatas (não com correlação) o que a etapa 1 apontou: colunas derivadas, constantes, variações de escrita e valores fora do domínio. São verificações estruturais, sem nenhum parâmetro aprendido para o modelo.

In [2]:
# Membro 1 — 1.2 Evidências estruturais: redundâncias, constantes, categorias, domínio
# Verificações determinísticas (igualdades e contagens). Não geram parâmetros usados pelo modelo.
def _rel(nome, original, derivada):
    ambos = original.notna() & derivada.notna()
    return {
        "derivada": nome,
        "relacao_testada": COLS_DERIVADAS[nome][1],
        "linhas_comparadas": int(ambos.sum()),
        "max_diferenca_abs": float((original[ambos] - derivada[ambos]).abs().max()),
        "mesmo_padrao_ausencia": bool(original.isna().equals(derivada.isna())),
        "ausentes_recuperaveis": int((original.isna() & derivada.notna()).sum()),
    }

Xe = X_treino
evid_redund = pd.DataFrame([
    _rel("src_bytes_kb", Xe["src_bytes"] / 1024, Xe["src_bytes_kb"]),
    _rel("dst_bytes_kb", Xe["dst_bytes"] / 1024, Xe["dst_bytes_kb"]),
    _rel("total_bytes", Xe["src_bytes"] + Xe["dst_bytes"], Xe["total_bytes"]),
    _rel("same_srv_pct", Xe["same_srv_rate"] * 100, Xe["same_srv_pct"]),
]).set_index("derivada")
# protocolo_cod: cada protocolo (normalizado) deve mapear para exatamente um código.
tab_proto = pd.crosstab(Xe["protocol_type"].str.strip().str.lower(), Xe["protocolo_cod"])
evid_redund.loc["protocolo_cod"] = {
    "relacao_testada": COLS_DERIVADAS["protocolo_cod"][1],
    "linhas_comparadas": int(Xe["protocol_type"].notna().sum()),
    "max_diferenca_abs": float(((tab_proto > 0).sum(axis=1) != 1).sum()),  # protocolos com >1 código
    "mesmo_padrao_ausencia": bool(Xe["protocol_type"].isna().equals(Xe["protocolo_cod"].isna())),
    "ausentes_recuperaveis": int((Xe["protocol_type"].isna() & Xe["protocolo_cod"].notna()).sum()),
}
print("Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):")
display(evid_redund)
print("protocol_type normalizado x protocolo_cod:")
display(tab_proto)

constantes = [c for c in Xe.columns if Xe[c].nunique(dropna=True) <= 1]
print("Atributos constantes no treino completo:", constantes)

evid_cat = pd.DataFrame({
    "categorias_brutas": Xe[COLS_CATEGORICAS].nunique(),
    "categorias_normalizadas": [Xe[c].str.strip().str.lower().nunique() for c in COLS_CATEGORICAS],
    "ausentes": Xe[COLS_CATEGORICAS].isna().sum(),
})
print("Inconsistência de escrita nos categóricos (caixa/espaços):")
display(evid_cat)

fora = pd.DataFrame({c: (Xe[c] < lo) | (Xe[c] > hi) for c, (lo, hi) in LIMITES_DOMINIO.items()})
print(f"Células fora do domínio: {int(fora.values.sum()):,}; linhas afetadas: {int(fora.any(axis=1).sum()):,}")

Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):


,relacao_testada,linhas_comparadas,max_diferenca_abs,mesmo_padrao_ausencia,ausentes_recuperaveis
derivada,,,,,
src_bytes_kb,src_bytes / 1024,41927,0.000000e+00,True,0
dst_bytes_kb,dst_bytes / 1024,69833,0.000000e+00,True,0
total_bytes,src_bytes + dst_bytes,41927,0.000000e+00,True,0
same_srv_pct,same_srv_rate * 100,66262,3.552714e-15,True,0
protocolo_cod,código 1:1 de protocol_type,69833,0.000000e+00,True,0


protocol_type normalizado x protocolo_cod:


protocolo_cod,0,1,2
protocol_type,,,
icmp,0,0,1643
tcp,56010,0,0
udp,0,12180,0


Atributos constantes no treino completo: ['num_outbound_cmds']
Inconsistência de escrita nos categóricos (caixa/espaços):


,categorias_brutas,categorias_normalizadas,ausentes
protocol_type,6,3,0
service,96,67,4416
flag,19,11,4093


Células fora do domínio: 6,121; linhas afetadas: 5,650


### 1.3 Transformações reutilizáveis

Cada etapa é um transformador do scikit-learn, encaixado em um `Pipeline`. Assim, medianas, modas, categorias do One-Hot e constantes são aprendidas **apenas no `fit`**, sempre chamado com o treino da dobra.

| Transformação | Aprende dos dados? | Opções comparadas |
|---|---|---|
| `NormalizarCategorias` | Não (strip + minúsculas) | brutas × normalizadas |
| `TratarForaDominio` | Não (limites da documentação) | manter × virar ausente × recortar |
| `RemoverRedundantes` | Sim: constantes da dobra | manter × remover derivadas + constantes |
| `SimpleImputer` numérico | Sim: mediana da dobra | mediana × mediana + indicador de ausência |
| `SimpleImputer` categórico | Sim: moda da dobra (se usada) | categoria `__ausente__` × moda |
| `filtrar_linhas_treino` | Sim: grupos do treino da dobra | todas × sem duplicatas × sem conflitos × ambos |

O One-Hot com `handle_unknown="ignore"` é só o mínimo para o modelo rodar. A comparação de codificações e escalas fica com o Membro 3.

In [3]:
# Membro 1 — 1.3 Transformações reutilizáveis (compatíveis com Pipeline do scikit-learn)
# Regra: tudo que aprende algo dos dados (constantes, medianas, modas, categorias) só aprende no fit,
# que é chamado apenas com o treino de cada dobra. O teste oficial só passa por transform.

class NormalizarCategorias(BaseEstimator, TransformerMixin):
    """Remove espaços e uniformiza caixa ('TCP ' -> 'tcp'). Sem estado; NaN continua NaN."""

    def __init__(self, colunas=tuple(COLS_CATEGORICAS), ativo=True):
        self.colunas = colunas
        self.ativo = ativo

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.colunas:
            if c in X:
                serie = X[c].astype(object)
                X[c] = serie.str.strip().str.lower() if self.ativo else serie
        return X


class TratarForaDominio(BaseEstimator, TransformerMixin):
    """Valores impossíveis pelos limites documentados.

    modo='manter' (referência), 'ausente' (vira NaN e segue para a imputação)
    ou 'recorte' (clip nos limites). Os limites vêm da documentação, não dos dados.
    """

    def __init__(self, limites=None, modo="manter"):
        self.limites = limites
        self.modo = modo

    def fit(self, X, y=None):
        assert self.modo in {"manter", "ausente", "recorte"}
        return self

    def transform(self, X):
        if self.modo == "manter":
            return X
        X = X.copy()
        for c, (lo, hi) in (self.limites or LIMITES_DOMINIO).items():
            if c not in X:
                continue
            if self.modo == "ausente":
                X[c] = X[c].astype(float).mask((X[c] < lo) | (X[c] > hi))
            else:
                X[c] = X[c].clip(lower=lo, upper=hi)
        return X


class RemoverRedundantes(BaseEstimator, TransformerMixin):
    """Remove derivadas determinísticas (lista fixa) e constantes detectadas NO TREINO DA DOBRA."""

    def __init__(self, derivadas=tuple(COLS_DERIVADAS), ativo=True):
        self.derivadas = derivadas
        self.ativo = ativo

    def fit(self, X, y=None):
        if not self.ativo:
            self.remover_ = []
            return self
        constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
        self.constantes_ = constantes
        self.remover_ = sorted(set(self.derivadas).intersection(X.columns) | set(constantes))
        return self

    def transform(self, X):
        return X.drop(columns=self.remover_)


def _eh_categorica(X):
    return [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]


def _eh_numerica(X):
    return [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]


def montar_preprocessamento(normalizar_categorias=True, fora_dominio="manter",
                            remover_redundantes=False, imputacao_num="mediana",
                            imputacao_cat="constante"):
    """Bloco de pré-processamento do Membro 1 (+ One-Hot mínimo, provisório até o Membro 3).

    imputacao_num: 'mediana' | 'mediana+indicador'
    imputacao_cat: 'constante' (ausência vira a categoria '__ausente__') | 'moda'
    """
    imp_num = SimpleImputer(strategy="median",
                            add_indicator=(imputacao_num == "mediana+indicador"))
    imp_cat = (SimpleImputer(strategy="constant", fill_value="__ausente__")
               if imputacao_cat == "constante" else SimpleImputer(strategy="most_frequent"))
    colunas = ColumnTransformer([
        ("num", imp_num, _eh_numerica),
        ("cat", Pipeline([("imp", imp_cat),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         _eh_categorica),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ("categorias", NormalizarCategorias(ativo=normalizar_categorias)),
        ("dominio", TratarForaDominio(modo=fora_dominio)),
        ("redundantes", RemoverRedundantes(ativo=remover_redundantes)),
        ("imputacao_codificacao", colunas),
    ])


def chave_registro(X):
    """Identificador de registros com atributos idênticos (categorias normalizadas, NaN como valor)."""
    Xn = NormalizarCategorias().transform(X)
    return pd.Series(pd.factorize(pd.util.hash_pandas_object(Xn, index=False))[0], index=X.index)


def filtrar_linhas_treino(X, y, estrategia="todas"):
    """Duplicatas/conflitos tratados SOMENTE no treino da dobra; a validação fica intacta.

    'todas'                      : mantém tudo (referência)
    'sem_duplicatas'             : remove cópias com mesmos atributos e mesma classe (mantém a 1ª)
    'sem_conflitos'              : remove TODOS os registros de grupos com atributos iguais e
                                   classes divergentes (não inventa nem troca rótulos)
    'sem_duplicatas_e_conflitos' : as duas remoções
    """
    if estrategia == "todas":
        return X, y
    if estrategia not in {"sem_duplicatas", "sem_conflitos", "sem_duplicatas_e_conflitos"}:
        raise ValueError(estrategia)
    chave = chave_registro(X)
    manter = pd.Series(True, index=X.index)
    if "duplicatas" in estrategia:
        manter &= ~pd.DataFrame({"k": chave, "y": y}).duplicated()
    if "conflitos" in estrategia:
        manter &= ~chave.map(y.groupby(chave).nunique()).gt(1)
    return X[manter], y[manter]

### 1.4 Dobras sem contaminação por duplicatas

Com `StratifiedKFold` comum, cópias idênticas de um registro caem em dobras diferentes, e a validação passa a medir memorização. Usamos `StratifiedGroupKFold`, com grupo = registro de atributos idênticos (categorias normalizadas, `NaN` tratado como valor). A auditoria abaixo compara as duas divisões. `INDICES_DOBRAS` fica disponível para os demais membros usarem exatamente as mesmas dobras.

In [4]:
# Membro 1 — 1.4 Dobras sem contaminação por cópias
# Registros com atributos idênticos recebem o mesmo grupo e ficam sempre na mesma dobra.
grupos_treino = chave_registro(X_treino)


def auditar_dobras(divisor, nome, usa_grupos):
    linhas = []
    args = (X_treino, y_treino, grupos_treino) if usa_grupos else (X_treino, y_treino)
    for i, (itr, iva) in enumerate(divisor.split(*args), start=1):
        chaves_tr = set(grupos_treino.iloc[itr])
        copia_no_treino = grupos_treino.iloc[iva].isin(chaves_tr)
        linhas.append({"divisao": nome, "dobra": i, "n_validacao": len(iva),
                       "pct_anomaly_validacao": 100 * y_treino.iloc[iva].mean(),
                       "pct_validacao_com_copia_no_treino": 100 * copia_no_treino.mean()})
    return linhas


DOBRAS = StratifiedGroupKFold(n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE)
auditoria = pd.DataFrame(
    auditar_dobras(StratifiedKFold(N_DOBRAS, shuffle=True, random_state=SEMENTE), "StratifiedKFold", False)
    + auditar_dobras(DOBRAS, "StratifiedGroupKFold (grupos = registro)", True)
)
print(f"Grupos de registros distintos: {grupos_treino.nunique():,} para {len(grupos_treino):,} linhas")
display(auditoria.groupby("divisao")[["n_validacao", "pct_anomaly_validacao",
                                      "pct_validacao_com_copia_no_treino"]].agg(["mean", "min", "max"]))
# Estrutura dos conflitos (atributos idênticos, classes diferentes) no treino completo — só diagnóstico.
perfil_grupos = y_treino.groupby(grupos_treino).agg(["size", "sum", "nunique"])
conflitos = perfil_grupos[perfil_grupos["nunique"] > 1]
print(f"Grupos conflitantes: {len(conflitos):,}; linhas: {int(conflitos['size'].sum()):,}; "
      f"anomalias envolvidas: {int(conflitos['sum'].sum()):,} de {int(y_treino.sum()):,} "
      f"({100 * conflitos['sum'].sum() / y_treino.sum():.1f}%)")
print("Composição dos grupos conflitantes (linhas x anomalias no grupo):")
display(pd.crosstab(conflitos["size"].rename("linhas_no_grupo"), conflitos["sum"].rename("anomalias_no_grupo")))

# Índices fixos das dobras, para os demais membros reutilizarem exatamente as mesmas divisões.
INDICES_DOBRAS = list(DOBRAS.split(X_treino, y_treino, grupos_treino))

Grupos de registros distintos: 56,766 para 69,833 linhas


n_validacao                \
                                                mean    min    max   
divisao                                                              
StratifiedGroupKFold (grupos = registro)     13966.6  13966  13968   
StratifiedKFold                              13966.6  13966  13967   

                                         pct_anomaly_validacao            \
                                                          mean       min   
divisao                                                                    
StratifiedGroupKFold (grupos = registro)              5.841078  5.835601   
StratifiedKFold                                       5.841078  5.835601   

                                                    \
                                               max   
divisao                                              
StratifiedGroupKFold (grupos = registro)  5.842761   
StratifiedKFold                           5.842761   

                                         pct_validacao_com_copia_no_treino  \
                                                                      mean   
divisao                                                                      
StratifiedGroupKFold (grupos = registro)                          0.000000   
StratifiedKFold                                                  29.709452   

                                                                
                                                min        max  
divisao                                                         
StratifiedGroupKFold (grupos = registro)   0.000000   0.000000  
StratifiedKFold                           29.505262  29.956326

Grupos conflitantes: 1,702; linhas: 3,730; anomalias envolvidas: 1,896 de 4,079 (46.5%)
Composição dos grupos conflitantes (linhas x anomalias no grupo):


anomalias_no_grupo,1,2
linhas_no_grupo,,
2,1376,0
3,132,194


### 1.5 Comparação com a referência (um fator por vez)

- **R0 (referência):** categorias brutas, valores fora do domínio mantidos, mediana nos numéricos, categoria `__ausente__` nos categóricos, todas as colunas e todas as linhas.
- Cada variação altera **um único fator** em relação a R0.
- Modelo provisório: `DecisionTreeClassifier(random_state=42)`. Árvores não dependem de escala, o que isola os efeitos desta seção do trabalho do Membro 3.
- Métrica principal: F1-macro na validação de cada dobra. A validação **nunca** é filtrada (duplicatas e conflitos continuam nela), então a estimativa reflete dados com o mesmo ruído.

In [5]:
# Membro 1 — 1.5 Comparação das estratégias (um fator por vez contra a referência)
# Modelo de referência provisório: árvore de decisão (insensível a escala, isola os efeitos do Membro 1).
# Mesmo modelo, mesma semente e mesmas dobras para todas as estratégias.
MODELO_REFERENCIA = DecisionTreeClassifier(random_state=SEMENTE)

REFERENCIA = dict(normalizar_categorias=False, fora_dominio="manter", remover_redundantes=False,
                  imputacao_num="mediana", imputacao_cat="constante", linhas_treino="todas")
VARIACOES = {
    "R0 referência": ("—", {}),
    "A1 categorias normalizadas": ("categorias", {"normalizar_categorias": True}),
    "B1 categórico: moda": ("ausentes categóricos", {"imputacao_cat": "moda"}),
    "B2 numérico: mediana + indicador": ("ausentes numéricos", {"imputacao_num": "mediana+indicador"}),
    "C1 fora do domínio -> ausente": ("inconsistências de domínio", {"fora_dominio": "ausente"}),
    "C2 fora do domínio -> recorte": ("inconsistências de domínio", {"fora_dominio": "recorte"}),
    "D1 remover derivadas e constantes": ("redundância de atributos", {"remover_redundantes": True}),
    "E1 treino sem duplicatas": ("redundância de registros", {"linhas_treino": "sem_duplicatas"}),
    "E2 treino sem conflitos": ("redundância de registros", {"linhas_treino": "sem_conflitos"}),
    "E3 treino sem duplicatas e conflitos": ("redundância de registros",
                                             {"linhas_treino": "sem_duplicatas_e_conflitos"}),
}


def avaliar_estrategia(config, modelo=MODELO_REFERENCIA, dobras=INDICES_DOBRAS):
    """Ajusta pré-processamento + modelo só no treino de cada dobra e avalia na validação intacta."""
    config = dict(config)
    linhas_treino = config.pop("linhas_treino")
    resultados = []
    for i, (itr, iva) in enumerate(dobras, start=1):
        X_tr, y_tr = filtrar_linhas_treino(X_treino.iloc[itr], y_treino.iloc[itr], linhas_treino)
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]
        pipe = Pipeline([("prep", montar_preprocessamento(**config)), ("modelo", clone(modelo))])
        t0 = time.perf_counter()
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_va)
        f1_normal, f1_anomaly = f1_score(y_va, pred, average=None, labels=[0, 1])
        resultados.append({
            "dobra": i, "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal, "f1_anomaly": f1_anomaly,
            "n_linhas_treino": len(X_tr), "n_atributos_finais": pipe[:-1].transform(X_va.head(1)).shape[1],
            "segundos": time.perf_counter() - t0,
        })
    return pd.DataFrame(resultados)


por_dobra = []
for nome, (fator, mudanca) in VARIACOES.items():
    res = avaliar_estrategia({**REFERENCIA, **mudanca})
    res.insert(0, "estrategia", nome)
    res.insert(1, "fator", fator)
    por_dobra.append(res)
por_dobra_m1 = pd.concat(por_dobra, ignore_index=True)

ref_f1 = por_dobra_m1.loc[por_dobra_m1.estrategia == "R0 referência"].set_index("dobra")["f1_macro"]
por_dobra_m1["delta_vs_R0"] = por_dobra_m1["f1_macro"] - por_dobra_m1["dobra"].map(ref_f1)

tabela_m1 = por_dobra_m1.groupby(["estrategia", "fator"], sort=False).agg(
    f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
    f1_anomaly=("f1_anomaly", "mean"), f1_normal=("f1_normal", "mean"),
    delta_vs_R0=("delta_vs_R0", "mean"),
    dobras_melhores_que_R0=("delta_vs_R0", lambda d: int((d > 0).sum())),
    linhas_treino=("n_linhas_treino", "mean"), atributos=("n_atributos_finais", "mean"),
    segundos=("segundos", "mean"),
).reset_index()
with pd.option_context("display.float_format", "{:.4f}".format):
    display(tabela_m1)
    print("F1-macro por dobra:")
    display(por_dobra_m1.pivot(index="estrategia", columns="dobra", values="f1_macro")
            .reindex(VARIACOES))

,estrategia,fator,f1_macro_media,f1_macro_dp,f1_anomaly,f1_normal,delta_vs_R0,dobras_melhores_que_R0,linhas_treino,atributos,segundos
0,R0 referência,—,0.8049,0.0054,0.6297,0.9800,0.0000,0,55866.4000,162.0000,2.0573
1,A1 categorias normalizadas,categorias,0.8027,0.0072,0.6255,0.9799,-0.0021,1,55866.4000,125.2000,1.7143
2,B1 categórico: moda,ausentes categóricos,0.8055,0.0082,0.6311,0.9800,0.0007,2,55866.4000,160.0000,2.2125
3,B2 numérico: mediana + indicador,ausentes numéricos,0.8048,0.0035,0.6296,0.9800,-0.0001,2,55866.4000,187.0000,2.3059
4,C1 fora do domínio -> ausente,inconsistências de domínio,0.8051,0.0043,0.6302,0.9800,0.0003,3,55866.4000,162.0000,2.3018
5,C2 fora do domínio -> recorte,inconsistências de domínio,0.8040,0.0063,0.6280,0.9799,-0.0009,1,55866.4000,162.0000,2.0437
6,D1 remover derivadas e constantes,redundância de atributos,0.8055,0.0056,0.6310,0.9800,0.0006,2,55866.4000,155.8000,1.9908
7,E1 treino sem duplicatas,redundância de registros,0.7817,0.0094,0.5847,0.9787,-0.0232,0,46774.4000,162.0000,2.2971
8,E2 treino sem conflitos,redundância de registros,0.8717,0.0041,0.7587,0.9847,0.0668,5,52882.4000,157.4000,1.7315
9,E3 treino sem duplicatas e conflitos,redundância de registros,0.8698,0.0046,0.7552,0.9845,0.0650,5,44051.2000,157.4000,1.3148


F1-macro por dobra:


dobra,1,2,3,4,5
estrategia,,,,,
R0 referência,0.8059,0.8106,0.8084,0.8028,0.7966
A1 categorias normalizadas,0.7963,0.8036,0.8074,0.8116,0.7948
B1 categórico: moda,0.7965,0.8092,0.8179,0.8010,0.8032
B2 numérico: mediana + indicador,0.8012,0.8089,0.8022,0.8036,0.8082
C1 fora do domínio -> ausente,0.8123,0.8030,0.8009,0.8050,0.8045
C2 fora do domínio -> recorte,0.8017,0.8044,0.8051,0.8131,0.7957
D1 remover derivadas e constantes,0.7957,0.8099,0.8069,0.8076,0.8073
E1 treino sem duplicatas,0.7662,0.7839,0.7860,0.7811,0.7912
E2 treino sem conflitos,0.8702,0.8724,0.8784,0.8681,0.8693


### 1.6 Combinação proposta e compatibilidade com o teste

Juntamos as opções escolhidas (justificadas na tabela abaixo) e conferimos que o pipeline ajustado **somente no treino** transforma o teste oficial sem remover nem reordenar linhas e sem deixar ausentes. Nenhuma estatística do teste é calculada e nenhuma previsão é gerada.

In [6]:
# Membro 1 — 1.6 Combinação proposta e checagem de compatibilidade com o teste oficial
# Opções escolhidas pelos resultados da 1.5 e por justificativa técnica (ver tabela 1.7).
# Atenção: escolher e reavaliar nas mesmas dobras é otimista; o Membro 5 deve confirmar na integração.
CONFIG_M1 = dict(normalizar_categorias=True, fora_dominio="ausente", remover_redundantes=True,
                 imputacao_num="mediana", imputacao_cat="constante", linhas_treino="sem_conflitos")

res_comb = avaliar_estrategia(CONFIG_M1)
res_comb.insert(0, "estrategia", "F1 combinação proposta")
res_comb["delta_vs_R0"] = res_comb["f1_macro"] - res_comb["dobra"].map(ref_f1)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(res_comb)
print(f"F1-macro combinação: {res_comb.f1_macro.mean():.4f} ± {res_comb.f1_macro.std():.4f} "
      f"(R0: {ref_f1.mean():.4f}); melhor que R0 em {(res_comb.delta_vs_R0 > 0).sum()}/{N_DOBRAS} dobras")

# Compatibilidade: ajusta SÓ no treino e transforma o teste sem remover/reordenar linhas.
# Nenhuma estatística do teste é usada; nenhuma previsão é gerada aqui.
config_prep = {k: v for k, v in CONFIG_M1.items() if k != "linhas_treino"}
X_fit, y_fit = filtrar_linhas_treino(X_treino, y_treino, CONFIG_M1["linhas_treino"])
prep_m1 = montar_preprocessamento(**config_prep).fit(X_fit, y_fit)
ids_teste, X_teste = carregar_teste()
assert list(X_teste.columns) == list(X_treino.columns), "Esquema do teste difere do treino."
Z_teste = prep_m1.transform(X_teste)
assert Z_teste.shape[0] == len(ids_teste) and not np.isnan(Z_teste).any()
print(f"Teste transformado: {Z_teste.shape[0]:,} linhas (ordem e IDs preservados), "
      f"{Z_teste.shape[1]} colunas, sem ausentes.")
print("Colunas removidas no ajuste:", prep_m1.named_steps["redundantes"].remover_)

,estrategia,dobra,f1_macro,f1_normal,f1_anomaly,n_linhas_treino,n_atributos_finais,segundos,delta_vs_R0
0,F1 combinação proposta,1,0.8685,0.9841,0.7528,52884,117,1.2562,0.0626
1,F1 combinação proposta,2,0.8703,0.9845,0.7561,52879,117,1.2805,0.0597
2,F1 combinação proposta,3,0.8779,0.9853,0.7705,52864,117,1.6289,0.0695
3,F1 combinação proposta,4,0.8664,0.9843,0.7485,52891,117,1.3792,0.0636
4,F1 combinação proposta,5,0.8701,0.9845,0.7557,52894,117,1.3394,0.0734


F1-macro combinação: 0.8706 ± 0.0044 (R0: 0.8049); melhor que R0 em 5/5 dobras
Teste transformado: 24,333 linhas (ordem e IDs preservados), 118 colunas, sem ausentes.
Colunas removidas no ajuste: ['dst_bytes_kb', 'num_outbound_cmds', 'protocolo_cod', 'same_srv_pct', 'src_bytes_kb', 'total_bytes']


### Resultados e justificativa — Membro 1

**Evidências estruturais (1.2).** As cinco colunas derivadas reproduzem exatamente a coluna de origem: `src_bytes_kb` = `src_bytes`/1024, `dst_bytes_kb` = `dst_bytes`/1024, `total_bytes` = `src_bytes` + `dst_bytes`, `same_srv_pct` = 100 × `same_srv_rate` (diferença máxima 4e-15, erro de ponto flutuante) e `protocolo_cod` 1:1 com `protocol_type` normalizado. Todas têm **o mesmo padrão de ausência** da origem, então não recuperam nenhum ausente. São redundância pura. `num_outbound_cmds` é constante. Os categóricos caem de 6→3, 96→67 e 19→11 categorias após normalizar caixa e espaços. Há 6.121 células fora do domínio, em 5.650 linhas.

**Duplicatas e conflitos (1.4).** As 69.833 linhas formam 56.766 registros distintos. Com `StratifiedKFold`, **~29,7% das linhas de validação têm uma cópia idêntica no treino da dobra**; com `StratifiedGroupKFold` agrupado por registro, 0%, mantendo 5,84% de `anomaly` em todas as dobras. Os conflitos são o achado mais relevante. São 1.702 grupos (3.730 linhas) com atributos idênticos e classes opostas, quase todos pares 1 normal + 1 anomaly. Eles concentram **1.896 das 4.079 anomalias (46,5%)**. O padrão sugere ruído de rótulo (cópias com rótulo trocado), mas não temos como saber qual rótulo é o verdadeiro.

**Comparação (1.5), F1-macro médio em 5 dobras (R0 = 0,8049 ± 0,0054):**

| Fator | Estratégias comparadas | Evidência (Δ F1-macro vs R0; dobras melhores) | Decisão e justificativa |
|---|---|---|---|
| Categorias inconsistentes | brutas × normalizadas | −0,0021; 1/5 (dentro do ruído, dp ≈ 0,005–0,007) | **Normalizar.** Desempenho equivalente com 162 → 125 colunas; sem isso, `TCP` e `tcp` seriam categorias diferentes, e uma variação nova no teste viraria "desconhecida". |
| Ausentes categóricos | `__ausente__` × moda | +0,0007; 2/5 (empate) | **Categoria `__ausente__`.** Empate no desempenho; mantém a ausência visível em vez de inventar o valor mais comum. |
| Ausentes numéricos | mediana × mediana + indicador | −0,0001; 2/5 (empate) | **Mediana sem indicador** para a árvore (+25 colunas sem ganho). Como `dst_host_serror_rate` tem ausência informativa (etapa 1), reavaliar o indicador com modelos lineares/MLP. |
| Valores fora do domínio | manter × ausente × recorte | ausente +0,0003 (3/5); recorte −0,0009 (1/5) | **Tratar como ausente.** Pequena vantagem e coerência: um valor impossível é erro de registro, e recortar inventaria um valor no limite. |
| Redundância de atributos | manter × remover derivadas + constantes | +0,0006; 2/5 | **Remover.** Igualdade exata comprovada, nenhum ausente recuperável; reduz 6 colunas sem perda. Constantes são detectadas no treino de cada dobra. |
| Duplicatas no treino | manter × remover | −0,0232; 0/5 | **Manter.** Remover cópias muda o peso das amostras e piora todas as dobras; a contaminação já é evitada pelo agrupamento das dobras. |
| Conflitos no treino | manter × remover grupos conflitantes | **+0,0668; 5/5** (0,8717 ± 0,0041; F1 anomaly 0,630 → 0,759) | **Remover só do treino de cada dobra**, sem trocar rótulos. A validação continua com os conflitos, então o ganho não vem de facilitar a avaliação. Remover também as duplicatas (E3) não acrescenta nada (0,8698). |

**Combinação proposta (1.6):** categorias normalizadas, fora do domínio → ausente, remoção de derivadas e constantes, mediana + `__ausente__` e treino sem conflitos. Resultado: **F1-macro 0,8706 ± 0,0044**, melhor que R0 nas 5 dobras, com 117 colunas (R0: 162). O ganho vem quase todo dos conflitos; os demais fatores são neutros em desempenho e foram escolhidos por coerência e menor dimensão. O pipeline ajustado só no treino transforma as 24.333 linhas do teste sem perdas, sem reordenação e sem ausentes.

**Limitações e pendências para a integração (Membro 5).**
- As opções foram escolhidas e reavaliadas nas mesmas dobras, o que torna a estimativa da combinação um pouco otimista. Confirmar com outra semente de dobras e com os modelos da etapa 3.
- O efeito dos conflitos foi medido com uma árvore; conferir se persiste com os demais modelos e com o balanceamento do Membro 2.
- O agrupamento considera iguais apenas registros idênticos, inclusive nos `NaN`. Cópias que diferem só por um valor ausente não são agrupadas.
- Os sinais operacionais estão excluídos até a conclusão do Membro 5.
- Nenhuma decisão usou o teste; o teste oficial não foi alterado.

## Membro 2 — Duas técnicas de balanceamento

**Tarefa:** Escolher e implementar pelo menos duas técnicas, comparando-as com nenhum balanceamento nas mesmas dobras e com o mesmo modelo de referência. Exemplos: SMOTE versus RandomUnderSampler, ou RandomOverSampler versus RandomUnderSampler. Justificar o tratamento de categorias caso use método sintético.

**Entregável:** Comparação de F1-macro e métricas por classe, distribuição antes/depois apenas no treino de cada dobra e discussão de perda de informação/sobreajuste.


### 2.1 Protocolo e escolha dos balanceamentos

Comparamos **sem balanceamento**, **RandomOverSampler (ROS)**, **RandomUnderSampler (RUS)** e **SMOTENC + Tomek Links**, usando `INDICES_DOBRAS` e a configuração `CONFIG_M1` (inclusive remoção de conflitos somente no treino). A árvore da parte 1 continua como referência; o **XGBoost** é um experimento adicional, com os mesmos balanceamentos e divisões. Assim, comparamos balanceamentos **dentro de cada modelo**, sem atribuir ao balanceamento uma melhora que pode vir do classificador.

A proposta SMOTETomek foi adaptada para dados mistos: **SMOTENC → One-Hot → TomekLinks**. Não usamos a classe `SMOTETomek` padrão: SMOTE sobre One-Hot pode interpolar indicadores e produzir categorias fracionárias. SMOTENC escolhe categorias existentes; o One-Hot posterior permite calcular os vizinhos de Tomek sem impor uma ordem artificial às categorias. Os indicadores discretos conhecidos também são tratados como categorias. Contagens numéricas ainda podem ficar fracionárias nos exemplos sintéticos: eles são vetores de treinamento, não conexões reais reconstruídas.

Imputação e `StandardScaler` são ajustados **antes da reamostragem, só no treino da dobra**. A escala é necessária para as distâncias de SMOTENC/Tomek e fica igual em todas as estratégias; isto não substitui a comparação de escalas do Membro 3. ROS e SMOTENC visam proporção 1:1; RUS reduz a maioria até 1:1, sem reposição; Tomek remove os dois lados dos pares (`sampling_strategy="all"`), reduzindo o total de linhas após a reamostragem. Esses parâmetros são fixados antes da comparação, sem busca nas dobras.

O XGBoost usa 200 árvores, profundidade 4, taxa 0,05 e limiar padrão 0,5; `scale_pos_weight=1` evita misturar ponderação com reamostragem. Não há early stopping orientado pela validação nem ajuste de hiperparâmetros. Validação e teste oficial nunca são balanceados; esta seção nem lê o teste. Os sinais operacionais continuam excluídos.

Fontes: [SMOTENC](https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTENC.html), [SMOTETomek](https://imbalanced-learn.org/stable/references/generated/imblearn.combine.SMOTETomek.html), [TomekLinks](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.TomekLinks.html) e [XGBClassifier](https://xgboost.readthedocs.io/en/stable/python/python_api.html#xgboost.XGBClassifier).

In [7]:
# Membro 2 — blocos reutilizáveis, com reamostragem apenas durante fit.
import platform
import sklearn
import imblearn
import xgboost
from imblearn.over_sampling import RandomOverSampler, SMOTENC
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.pipeline import Pipeline as PipelineBalanceado
from sklearn.metrics import precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits

print({"Python": platform.python_version(), "numpy": np.__version__,
       "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
       "imbalanced-learn": imblearn.__version__, "xgboost": xgboost.__version__})

COLS_NOMINAIS_M2 = tuple(COLS_CATEGORICAS) + (
    "land", "logged_in", "root_shell", "su_attempted", "is_host_login", "is_guest_login",
)
ESTRATEGIAS_M2 = ("Sem balanceamento", "ROS", "RUS", "SMOTENC + Tomek")
MODELOS_M2 = {
    "Árvore": clone(MODELO_REFERENCIA),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=1.0, colsample_bytree=1.0, scale_pos_weight=1,
        objective="binary:logistic", eval_metric="logloss", tree_method="hist",
        random_state=SEMENTE, n_jobs=2,
    ),
}


def contagens_m2(y):
    return pd.Series(np.asarray(y)).value_counts().reindex([0, 1], fill_value=0).astype(int).to_dict()


def nominais_m2(X):
    return [c for c in COLS_NOMINAIS_M2 if c in X.columns]


def numericas_m2(X):
    return [c for c in X.columns if c not in COLS_NOMINAIS_M2]


class CategoriasM2(BaseEstimator, TransformerMixin):
    """Converte indicadores discretos para texto, preservando NaN para a imputação."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in nominais_m2(X):
            X[c] = X[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
        return X


class AmostradorAuditadoM2(BaseEstimator):
    """Registra contagens antes/depois; não participa de predict/transform."""
    def __init__(self, amostrador=None):
        self.amostrador = amostrador

    def fit_resample(self, X, y):
        self.antes_ = contagens_m2(y)
        self.amostrador_ = clone(self.amostrador) if self.amostrador is not None else None
        Xr, yr = (X, y) if self.amostrador_ is None else self.amostrador_.fit_resample(X, y)
        self.depois_ = contagens_m2(yr)
        if isinstance(self.amostrador_, SMOTENC):
            # SMOTENC deve preservar os domínios categóricos aprendidos nesta dobra.
            for c in nominais_m2(X):
                assert set(Xr[c].unique()).issubset(set(X[c].unique())), c
        return Xr, yr


def montar_pipeline_m2(estrategia, modelo, config=None):
    """Pipeline reutilizável. O chamador filtra conflitos apenas no treino antes de fit.

    Todas as estatísticas são ajustadas no fit; os amostradores são pulados em predict.
    """
    if estrategia not in ESTRATEGIAS_M2:
        raise ValueError(estrategia)
    cfg = dict(CONFIG_M1 if config is None else config)
    if cfg["imputacao_num"] != "mediana" or cfg["imputacao_cat"] != "constante":
        raise ValueError("Este bloco usa mediana e categoria de ausência, conforme CONFIG_M1.")
    numericos = Pipeline([
        ("imputar", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("escala", StandardScaler()),
    ])
    misto = ColumnTransformer([
        ("num", numericos, numericas_m2),
        ("cat", SimpleImputer(strategy="constant", fill_value="__ausente__",
                              keep_empty_features=True), nominais_m2),
    ], verbose_feature_names_out=False).set_output(transform="pandas")
    # Os nomes sobrevivem à imputação; constantes podem ter sido removidas no fit.
    # SMOTENC recebe a máscara categórica por dtype, sem índices fixos frágeis.
    amostradores = {
        "Sem balanceamento": None,
        "ROS": RandomOverSampler(sampling_strategy=1.0, random_state=SEMENTE),
        "RUS": RandomUnderSampler(sampling_strategy=1.0, replacement=False, random_state=SEMENTE),
        "SMOTENC + Tomek": SMOTENC(categorical_features="auto", sampling_strategy=1.0,
                                   k_neighbors=5, random_state=SEMENTE),
    }
    codificar = ColumnTransformer([
        ("num", "passthrough", numericas_m2),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominais_m2),
    ])
    return PipelineBalanceado([
        ("categorias", NormalizarCategorias(ativo=cfg["normalizar_categorias"])),
        ("dominio", TratarForaDominio(modo=cfg["fora_dominio"])),
        ("redundantes", RemoverRedundantes(ativo=cfg["remover_redundantes"])),
        ("tipos", CategoriasM2()),
        ("imputar_escalar", misto),
        ("categorias_smote", DtypeCategoriasM2()),
        ("reamostrar", AmostradorAuditadoM2(amostradores[estrategia])),
        ("codificar", codificar),
        ("tomek", AmostradorAuditadoM2(
            TomekLinks(sampling_strategy="all", n_jobs=2) if estrategia == "SMOTENC + Tomek" else None)),
        ("modelo", clone(modelo)),
    ])


class DtypeCategoriasM2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in nominais_m2(X):
            X[c] = X[c].astype("category")
        return X

{'Python': '3.12.3', 'numpy': '1.26.4', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1'}


### 2.2 Execução nas cinco dobras comuns

Cada combinação recebe um pipeline novo. Registramos precisão, recall e F1 das duas classes, F1-macro, tempo e distribuições em quatro pontos: treino original, após filtro de conflitos, após reamostragem e após Tomek. As contagens da validação são registradas separadamente e permanecem iguais entre todas as combinações. O tempo inclui ajuste e predição e depende da máquina.

In [8]:
# Membro 2 — 4 estratégias × 2 modelos × 5 dobras, sem consultar o teste oficial.
def avaliar_balanceamentos_m2(config=None, dobras=None, modelos=None):
    cfg = dict(CONFIG_M1 if config is None else config)
    dobras = INDICES_DOBRAS if dobras is None else dobras
    modelos = MODELOS_M2 if modelos is None else modelos
    resultados, distribuicoes = [], []
    for dobra, (itr, iva) in enumerate(dobras, start=1):
        assert set(grupos_treino.iloc[itr]).isdisjoint(grupos_treino.iloc[iva])
        X_original, y_original = X_treino.iloc[itr], y_treino.iloc[itr]
        X_tr, y_tr = filtrar_linhas_treino(X_original, y_original, cfg["linhas_treino"])
        X_va, y_va = X_treino.iloc[iva].copy(), y_treino.iloc[iva].copy()
        for nome_modelo, modelo in modelos.items():
            for estrategia in ESTRATEGIAS_M2:
                pipe = montar_pipeline_m2(estrategia, modelo, cfg)
                inicio = time.perf_counter()
                # Limita threads numéricas para não saturar a máquina nos vizinhos mais próximos.
                with threadpool_limits(limits=2):
                    pipe.fit(X_tr, y_tr)
                    pred = pipe.predict(X_va)
                segundos = time.perf_counter() - inicio
                assert len(pred) == len(y_va)
                assert X_va.equals(X_treino.iloc[iva]) and y_va.equals(y_treino.iloc[iva])
                p, r, f, suporte = precision_recall_fscore_support(
                    y_va, pred, labels=[0, 1], zero_division=0)
                registro = {"modelo": nome_modelo, "estrategia": estrategia, "dobra": dobra,
                            "f1_macro": float(f.mean()), "segundos": segundos}
                for j, classe in enumerate(CLASSES):
                    registro.update({f"precisao_{classe}": p[j], f"recall_{classe}": r[j],
                                     f"f1_{classe}": f[j], f"suporte_{classe}": int(suporte[j])})
                resultados.append(registro)
                etapas = {"original": contagens_m2(y_original), "filtrado": contagens_m2(y_tr),
                          "reamostrado": pipe.named_steps["reamostrar"].depois_,
                          "final": pipe.named_steps["tomek"].depois_,
                          "validacao": contagens_m2(y_va)}
                dist = {"modelo": nome_modelo, "estrategia": estrategia, "dobra": dobra}
                for etapa, contagens in etapas.items():
                    for j, classe in enumerate(CLASSES):
                        dist[f"{etapa}_{classe}"] = contagens[j]
                distribuicoes.append(dist)
        print(f"Dobra {dobra}/{len(dobras)} concluída.", flush=True)
    return pd.DataFrame(resultados), pd.DataFrame(distribuicoes)


por_dobra_m2, distribuicoes_m2 = avaliar_balanceamentos_m2()
referencia_m2 = por_dobra_m2[por_dobra_m2.estrategia == "Sem balanceamento"]
por_dobra_m2 = por_dobra_m2.merge(
    referencia_m2[["modelo", "dobra", "f1_macro"]].rename(columns={"f1_macro": "f1_sem_balanceamento"}),
    on=["modelo", "dobra"], validate="many_to_one",
)
por_dobra_m2["delta_f1"] = por_dobra_m2.f1_macro - por_dobra_m2.f1_sem_balanceamento

# Auditoria: validação intacta, mesma distribuição entre modelos e 1:1 nos amostradores.
assert distribuicoes_m2.groupby("dobra")[["validacao_normal", "validacao_anomaly"]].nunique().eq(1).all().all()
cols_contagens_m2 = [c for c in distribuicoes_m2 if c not in ["modelo", "estrategia", "dobra"]]
assert distribuicoes_m2.groupby(["estrategia", "dobra"])[cols_contagens_m2].nunique().eq(1).all().all()
balanceadas_m2 = distribuicoes_m2[distribuicoes_m2.estrategia != "Sem balanceamento"]
assert balanceadas_m2.reamostrado_normal.eq(balanceadas_m2.reamostrado_anomaly).all()
sem_m2 = distribuicoes_m2[distribuicoes_m2.estrategia == "Sem balanceamento"]
assert sem_m2.filtrado_normal.eq(sem_m2.final_normal).all()
assert sem_m2.filtrado_anomaly.eq(sem_m2.final_anomaly).all()
assert distribuicoes_m2.final_normal.le(distribuicoes_m2.reamostrado_normal).all()
assert distribuicoes_m2.final_anomaly.le(distribuicoes_m2.reamostrado_anomaly).all()
print("Auditorias aprovadas: grupos disjuntos, categorias válidas e validação intacta.")

Dobra 1/5 concluída.
Dobra 2/5 concluída.
Dobra 3/5 concluída.
Dobra 4/5 concluída.
Dobra 5/5 concluída.
Auditorias aprovadas: grupos disjuntos, categorias válidas e validação intacta.


### 2.3 Resultados e distribuição das classes

O desvio padrão resume a variação entre as cinco dobras; não é um intervalo de confiança. `delta_f1` compara cada estratégia com **o mesmo modelo sem balanceamento na mesma dobra**. As distribuições são idênticas para os dois modelos, por isso são exibidas uma única vez. `distribuicoes_m2` conserva todas as contagens, inclusive antes da remoção de conflitos e após a geração das amostras sintéticas.

In [9]:
metricas_m2 = [f"{metrica}_{classe}" for classe in CLASSES for metrica in ["precisao", "recall", "f1"]]
tabela_m2 = por_dobra_m2.groupby(["modelo", "estrategia"], sort=False).agg(
    f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
    delta_f1=("delta_f1", "mean"),
    dobras_melhores=("delta_f1", lambda s: int((s > 0).sum())),
    segundos_media=("segundos", "mean"),
    **{metrica: (metrica, "mean") for metrica in metricas_m2},
).reset_index()
with pd.option_context("display.float_format", "{:.4f}".format, "display.max_columns", None):
    display(tabela_m2[["modelo", "estrategia", "f1_macro_media", "f1_macro_dp", "delta_f1",
                       "dobras_melhores", "segundos_media"]])
    display(tabela_m2[["modelo", "estrategia"] + metricas_m2])
    display(por_dobra_m2.pivot(index=["modelo", "estrategia"], columns="dobra", values="f1_macro"))

# Uma linha por dobra/estratégia, pois os dois modelos recebem as mesmas contagens.
dist_exibir_m2 = distribuicoes_m2[distribuicoes_m2.modelo == "Árvore"].copy()
dist_exibir_m2["removidos_tomek"] = (
    dist_exibir_m2.reamostrado_normal + dist_exibir_m2.reamostrado_anomaly
    - dist_exibir_m2.final_normal - dist_exibir_m2.final_anomaly
)
with pd.option_context("display.max_rows", 30, "display.max_columns", None):
    display(dist_exibir_m2.drop(columns="modelo"))

# Candidatos exploratórios para a integração; não são configurações finais aprovadas.
candidatos_m2 = tabela_m2.loc[tabela_m2.groupby("modelo").f1_macro_media.idxmax(),
                             ["modelo", "estrategia", "f1_macro_media"]].reset_index(drop=True)
display(candidatos_m2)

,modelo,estrategia,f1_macro_media,f1_macro_dp,delta_f1,dobras_melhores,segundos_media
0,Árvore,Sem balanceamento,0.8712,0.0043,0.0000,0,1.7290
1,Árvore,ROS,0.8665,0.0031,-0.0047,0,1.8168
2,Árvore,RUS,0.7991,0.0078,-0.0721,0,0.4863
3,Árvore,SMOTENC + Tomek,0.8630,0.0041,-0.0082,0,231.6400
4,XGBoost,Sem balanceamento,0.8705,0.0042,0.0000,0,1.5917
5,XGBoost,ROS,0.8597,0.0041,-0.0108,0,2.6396
6,XGBoost,RUS,0.8424,0.0033,-0.0281,0,0.5738
7,XGBoost,SMOTENC + Tomek,0.8619,0.0025,-0.0087,0,231.0283


,modelo,estrategia,precisao_normal,recall_normal,f1_normal,precisao_anomaly,recall_anomaly,f1_anomaly
0,Árvore,Sem balanceamento,0.9861,0.9830,0.9846,0.7397,0.7771,0.7579
1,Árvore,ROS,0.9850,0.9835,0.9842,0.7399,0.7580,0.7488
2,Árvore,RUS,0.9882,0.9515,0.9695,0.5117,0.8161,0.6288
3,Árvore,SMOTENC + Tomek,0.9858,0.9809,0.9834,0.7152,0.7725,0.7427
4,XGBoost,Sem balanceamento,0.9845,0.9854,0.9850,0.7617,0.7507,0.7561
5,XGBoost,ROS,0.9881,0.9759,0.9820,0.6761,0.8112,0.7374
6,XGBoost,RUS,0.9882,0.9695,0.9788,0.6236,0.8137,0.7060
7,XGBoost,SMOTENC + Tomek,0.9869,0.9786,0.9828,0.6969,0.7911,0.7410


dobra                          1      2      3      4      5
modelo  estrategia                                          
XGBoost ROS               0.8591 0.8615 0.8589 0.8651 0.8539
        RUS               0.8455 0.8454 0.8423 0.8376 0.8412
        SMOTENC + Tomek   0.8589 0.8641 0.8648 0.8604 0.8612
        Sem balanceamento 0.8670 0.8714 0.8756 0.8657 0.8730
Árvore  ROS               0.8630 0.8671 0.8710 0.8643 0.8672
        RUS               0.8071 0.8020 0.7867 0.8029 0.7969
        SMOTENC + Tomek   0.8597 0.8695 0.8592 0.8633 0.8635
        Sem balanceamento 0.8699 0.8715 0.8784 0.8679 0.8683

,estrategia,dobra,original_normal,original_anomaly,filtrado_normal,filtrado_anomaly,reamostrado_normal,reamostrado_anomaly,final_normal,final_anomaly,validacao_normal,validacao_anomaly,removidos_tomek
0,Sem balanceamento,1,52603,3263,51142,1742,51142,1742,51142,1742,13151,816,0
1,ROS,1,52603,3263,51142,1742,51142,51142,51142,51142,13151,816,0
2,RUS,1,52603,3263,51142,1742,1742,1742,1742,1742,13151,816,0
3,SMOTENC + Tomek,1,52603,3263,51142,1742,51142,51142,51116,51116,13151,816,52
8,Sem balanceamento,2,52602,3263,51128,1751,51128,1751,51128,1751,13152,816,0
9,ROS,2,52602,3263,51128,1751,51128,51128,51128,51128,13152,816,0
10,RUS,2,52602,3263,51128,1751,1751,1751,1751,1751,13152,816,0
11,SMOTENC + Tomek,2,52602,3263,51128,1751,51128,51128,51110,51110,13152,816,36
16,Sem balanceamento,3,52604,3263,51125,1739,51125,1739,51125,1739,13150,816,0
17,ROS,3,52604,3263,51125,1739,51125,51125,51125,51125,13150,816,0


,modelo,estrategia,f1_macro_media
0,XGBoost,Sem balanceamento,0.870542
1,Árvore,Sem balanceamento,0.871227


### Resultados e justificativa — Membro 2

**Resultado observado.** Foram executadas 40 avaliações (quatro estratégias × dois modelos × cinco dobras), com grupos disjuntos e validação intacta. F1-macro médio ± desvio padrão:

| Estratégia | Árvore de Decisão | XGBoost |
|---|---:|---:|
| Sem balanceamento | **0,8718 ± 0,0040** | **0,8705 ± 0,0042** |
| RandomOverSampler | 0,8668 ± 0,0033 | 0,8597 ± 0,0041 |
| RandomUnderSampler | 0,7991 ± 0,0078 | 0,8424 ± 0,0033 |
| SMOTENC + Tomek Links | 0,8633 ± 0,0041 | 0,8619 ± 0,0025 |

Nenhum dos três balanceamentos superou a referência sem balanceamento em nenhuma das cinco dobras, para os dois modelos. **A proposta para integração é manter, por enquanto, ausência de balanceamento.** A diferença entre árvore e XGBoost sem balanceamento é pequena diante da variação entre dobras; este experimento não estabelece superioridade geral de um algoritmo.

**XGBoost + SMOTENC/Tomek.** O recall de `anomaly` subiu de **0,7507 para 0,7911**, mas a precisão caiu de **0,7617 para 0,6969**, indicando maior cobertura de anomalias acompanhada de mais falsos positivos. O F1 da classe caiu de 0,7561 para 0,7410, e o F1-macro de 0,8705 para 0,8619 (Δ = −0,0087). Portanto, a combinação não é a escolha pelo critério F1-macro definido pelo trabalho. Se o objetivo operacional priorizasse recall, seria necessário definir o custo dos erros e validar esse critério separadamente. O custo médio também subiu: aproximadamente 15,91 s por dobra, contra 0,83 s sem balanceamento, incluindo ajuste e predição nesta máquina.

**Distribuição e efeitos dos métodos.** Na primeira dobra, o treino original tem 52.603 registros `normal` e 3.263 `anomaly`. Após remover grupos conflitantes apenas no treino, restam **51.142 normais e 1.742 anomalias** (3,29% de anomalias). A validação continua com **13.151 normais e 816 anomalias** em todas as estratégias.

- **ROS:** passa para 51.142 registros de cada classe, acrescentando 49.400 cópias de anomalias. Preserva os registros da maioria, mas não cria informação nova e aumenta o peso de poucos exemplos minoritários. Pode favorecer sobreajuste; os resultados não mostram ganho de F1-macro, mas não isolam sua causa.
- **RUS:** mantém 1.742 registros de cada classe. Descarta 49.400 normais, cerca de **96,6% da maioria** naquela dobra. É rápido, mas essa perda de cobertura pode prejudicar a identificação de tráfego normal. No XGBoost, o recall de anomalias cresce para 0,8137 e a precisão cai para 0,6236; a árvore apresenta a maior queda de F1-macro.
- **SMOTENC + Tomek:** gera 49.400 anomalias sintéticas na primeira dobra, respeitando os valores nominais/discretos observados no treino. Em seguida, Tomek remove 52 registros (26 de cada classe), deixando 51.116 de cada classe. Nas cinco dobras, remove entre 36 e 52 registros no total por dobra: a limpeza é pequena diante da expansão promovida pelo SMOTENC. A combinação exige mais tempo e pode interpolar vizinhos inadequados em regiões ruidosas; remover pares de Tomek não garante eliminar todo ruído nem identificar rótulos verdadeiros.

**Integração e limites.** `montar_pipeline_m2`, `por_dobra_m2`, `distribuicoes_m2`, `tabela_m2` e `candidatos_m2` ficam disponíveis para o Membro 5. Antes de `fit`, deve-se aplicar `filtrar_linhas_treino` somente ao treino da dobra, como no avaliador. As estatísticas de imputação e escala são aprendidas antes da reamostragem; os amostradores são ignorados durante `predict`.

As conclusões são condicionais à remoção de conflitos, aos parâmetros fixos, à proporção 1:1 e a uma única semente. Não houve busca por uma razão de balanceamento melhor. As mesmas dobras já orientaram escolhas da parte 1, então a seleção é exploratória e pode ser otimista; confirmar na integração com outra divisão por grupos. A referência desta seção inclui escala para distâncias e codificação dos indicadores discretos, por isso o efeito do balanceamento deve ser lido contra a referência **desta seção**, e não diretamente contra R0 da parte 1. SMOTENC + Tomek é uma adaptação para dados mistos, não a execução da classe `SMOTETomek` padrão. Confirmar com a equipe/professora se XGBoost atende ao requisito de algoritmo visto em aula antes de adotá-lo na etapa 3. O teste oficial não foi usado nesta comparação.

**Execução e uso de IA.** A parte 2 foi implementada e executada com apoio do OpenAI Codex, incluindo revisão do protocolo e das contagens. As versões utilizadas aparecem na célula 2.1 e as saídas desta seção estão salvas. Houve um aviso ambiental do `joblib` ao consultar núcleos físicos no macOS; a biblioteca usou a contagem de núcleos lógicos e a execução concluiu com todas as auditorias aprovadas. Os responsáveis devem revisar a interpretação e registrar suas contribuições efetivas antes da entrega; não se atribui aqui autoria individual não confirmada.

## Membro 3 — Codificação e transformação numérica

**Tarefa:** Implementar pelo menos uma conversão simbólico-numérica e uma transformação numérica. Justificar One-Hot ou alternativa para service e tratamento de categorias desconhecidas. Comparar com uma referência; por exemplo, sem escala versus StandardScaler/RobustScaler.

**Entregável:** Blocos de codificação/escala ajustados por dobra, dimensão resultante e comparação explícita com justificativa técnica.


### 3.1 Configuração dos experimentos

Usamos o carregamento, as cinco dobras e o modelo de referência definidos pelo Membro 1. Mantemos o tratamento dos dados fixo e não aplicamos balanceamento, para comparar o efeito da codificação e da transformação numérica. Todos os ajustes serão feitos apenas no treino de cada dobra.

In [10]:
print("Dados de treino:", X_treino.shape)
print("Quantidade de dobras:", len(INDICES_DOBRAS))
print("Modelo de referência:", MODELO_REFERENCIA)
print("Configuração proposta pelo Membro 1:")
display(pd.Series(CONFIG_M1, name="configuracao"))

config_base_m3 = CONFIG_M1.copy()

Dados de treino: (69833, 46)
Quantidade de dobras: 5
Modelo de referência: DecisionTreeClassifier(random_state=42)
Configuração proposta pelo Membro 1:


normalizar_categorias             True
fora_dominio                   ausente
remover_redundantes               True
imputacao_num                  mediana
imputacao_cat                constante
linhas_treino            sem_conflitos
Name: configuracao, dtype: object

### 3.2 Blocos de codificação e transformação numérica

Comparamos One-Hot sem agrupamento com One-Hot agrupando categorias com menos de 100 registros no treino de cada dobra. Para os atributos numéricos, comparamos ausência de escala, StandardScaler e RobustScaler. As demais escolhas do Membro 1 são mantidas. Como a árvore costuma ser pouco sensível à escala, resultados semelhantes entre as transformações não significam que elas sejam dispensáveis para outros modelos.

In [11]:
from sklearn.preprocessing import StandardScaler, RobustScaler

def montar_preprocessamento_m3(config, codificacao="onehot", escala="sem_escala"):
    parametros = {k: v for k, v in config.items() if k != "linhas_treino"}
    prep = montar_preprocessamento(**parametros)
    bloco = prep.named_steps["imputacao_codificacao"]

    # Mantém a imputação numérica e acrescenta a escala escolhida.
    imp_num = bloco.transformers[0][1]
    escaladores = {
        "sem_escala": "passthrough",
        "standard": StandardScaler(),
        "robust": RobustScaler()
    }
    numerico = Pipeline([("imp", imp_num), ("escala", escaladores[escala])])

    # Categorias e frequências serão aprendidas no fit de cada dobra.
    categorico = bloco.transformers[1][1]
    if codificacao == "onehot":
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    elif codificacao == "onehot_raras":
        encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False)
    else:
        raise ValueError("Codificação desconhecida.")

    categorico.set_params(ohe=encoder)
    novo_bloco = ColumnTransformer([
        ("num", numerico, _eh_numerica),
        ("cat", categorico, _eh_categorica)
    ], verbose_feature_names_out=False)

    prep.set_params(imputacao_codificacao=novo_bloco)
    return prep

print("Função de pré-processamento do Membro 3 definida.")

Função de pré-processamento do Membro 3 definida.


### 3.3 Comparação nas cinco dobras

Usamos One-Hot sem escala como referência. Alteramos um fator por vez: agrupamento de categorias raras ou escala numérica. Registramos F1-macro, F1 por classe, quantidade de colunas e tempo de execução. A validação permanece intacta e nenhum balanceamento é aplicado.

In [12]:
experimentos_m3 = {
    "One-Hot sem escala": ("onehot", "sem_escala"),
    "One-Hot com categorias raras agrupadas": ("onehot_raras", "sem_escala"),
    "One-Hot + StandardScaler": ("onehot", "standard"),
    "One-Hot + RobustScaler": ("onehot", "robust")
}

resultados_m3 = []

for nome, (codificacao, escala) in experimentos_m3.items():
    print(f"Executando: {nome}")

    for dobra, (itr, iva) in enumerate(INDICES_DOBRAS, start=1):
        X_tr, y_tr = filtrar_linhas_treino(
            X_treino.iloc[itr], y_treino.iloc[itr], config_base_m3["linhas_treino"]
        )
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]

        prep = montar_preprocessamento_m3(config_base_m3, codificacao, escala)
        pipeline_m3 = Pipeline([("prep", prep), ("modelo", clone(MODELO_REFERENCIA))])

        inicio = time.perf_counter()
        pipeline_m3.fit(X_tr, y_tr)
        previsoes = pipeline_m3.predict(X_va)
        segundos = time.perf_counter() - inicio

        f1_normal, f1_anomaly = f1_score(y_va, previsoes, labels=[0, 1], average=None, zero_division=0)
        n_colunas = pipeline_m3.named_steps["prep"].transform(X_va.head(1)).shape[1]

        resultados_m3.append({
            "experimento": nome, "dobra": dobra,
            "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal, "f1_anomaly": f1_anomaly,
            "n_colunas": n_colunas, "segundos": segundos
        })

por_dobra_m3 = pd.DataFrame(resultados_m3)

# Compara cada resultado com a referência na mesma dobra.
referencia_m3 = por_dobra_m3[
    por_dobra_m3["experimento"] == "One-Hot sem escala"
].set_index("dobra")["f1_macro"]

por_dobra_m3["delta_vs_referencia"] = (
    por_dobra_m3["f1_macro"] - por_dobra_m3["dobra"].map(referencia_m3)
)

tabela_m3 = por_dobra_m3.groupby("experimento", sort=False).agg(
    f1_macro_media=("f1_macro", "mean"),
    f1_macro_dp=("f1_macro", "std"),
    f1_normal=("f1_normal", "mean"),
    f1_anomaly=("f1_anomaly", "mean"),
    delta_vs_referencia=("delta_vs_referencia", "mean"),
    colunas_media=("n_colunas", "mean"),
    colunas_min=("n_colunas", "min"),
    colunas_max=("n_colunas", "max"),
    segundos_media=("segundos", "mean")
)

display(tabela_m3.round(4))
display(por_dobra_m3.pivot(index="experimento", columns="dobra", values="f1_macro").round(4))

Executando: One-Hot sem escala
Executando: One-Hot com categorias raras agrupadas
Executando: One-Hot + StandardScaler
Executando: One-Hot + RobustScaler


,f1_macro_media,f1_macro_dp,f1_normal,f1_anomaly,delta_vs_referencia,colunas_media,colunas_min,colunas_max,segundos_media
experimento,,,,,,,,,
One-Hot sem escala,0.8706,0.0044,0.9845,0.7567,0.0000,117.0,117,117,1.3021
One-Hot com categorias raras agrupadas,0.8730,0.0038,0.9848,0.7612,0.0024,65.8,65,66,0.8046
One-Hot + StandardScaler,0.8704,0.0045,0.9845,0.7562,-0.0003,117.0,117,117,1.2673
One-Hot + RobustScaler,0.8706,0.0043,0.9845,0.7566,-0.0000,117.0,117,117,1.2787


dobra,1,2,3,4,5
experimento,,,,,
One-Hot + RobustScaler,0.8685,0.8699,0.8779,0.8666,0.8701
One-Hot + StandardScaler,0.8682,0.8697,0.8779,0.8659,0.8701
One-Hot com categorias raras agrupadas,0.8715,0.8746,0.8788,0.8692,0.8709
One-Hot sem escala,0.8685,0.8703,0.8779,0.8664,0.8701


### 3.4 Categorias desconhecidas e limitações

No One-Hot comum, categorias desconhecidas são ignoradas e representadas por zeros nas colunas daquele atributo. Na alternativa com agrupamento, elas entram no grupo de categorias pouco frequentes, quando esse grupo existe; caso contrário, também são representadas por zeros. O limite de 100 registros é aplicado separadamente a cada atributo categórico e considera apenas o treino da dobra. O agrupamento pode incluir a categoria de ausência criada pela imputação, caso sua frequência fique abaixo desse limite. A demonstração abaixo usa valores fictícios, sem consultar o teste oficial.

O limite de 100 registros reaproveita o critério exploratório da primeira entrega e não foi escolhido buscando o melhor resultado nesta comparação. No exemplo fictício, usamos o limite de 2 apenas para demonstrar o funcionamento com poucos registros.

In [13]:
exemplo_treino_m3 = pd.DataFrame({"service": ["http", "http", "smtp", "smtp", "raro"]})
exemplo_novo_m3 = pd.DataFrame({"service": ["http", "servico_desconhecido"]})

encoders_exemplo_m3 = {
    "One-Hot comum": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "One-Hot com agrupamento": OneHotEncoder(
        handle_unknown="infrequent_if_exist", min_frequency=2, sparse_output=False
    )
}

for nome, encoder in encoders_exemplo_m3.items():
    encoder.fit(exemplo_treino_m3)
    transformado = encoder.transform(exemplo_novo_m3)
    print(f"\n{nome}:")
    display(pd.DataFrame(
        transformado,
        columns=encoder.get_feature_names_out(),
        index=exemplo_novo_m3["service"]
    ))


One-Hot comum:


,service_http,service_raro,service_smtp
service,,,
http,1.0,0.0,0.0
servico_desconhecido,0.0,0.0,0.0



One-Hot com agrupamento:


,service_http,service_smtp,service_infrequent_sklearn
service,,,
http,1.0,0.0,0.0
servico_desconhecido,0.0,0.0,1.0


### 3.5 Resultados e justificativa — Membro 3

O One-Hot sem escala apresentou F1-macro médio de 0,8706, com 117 colunas. Ao agrupar categorias com menos de 100 registros no treino de cada dobra, o F1-macro passou para 0,8730, com desvio padrão de 0,0038. A quantidade de colunas caiu para 65 a 66, pois as frequências variam entre os treinos das dobras. Essa alternativa superou a referência nas cinco dobras, mas a diferença média de 0,0024 é pequena e não permite afirmar que haverá ganho em dados futuros. O agrupamento fica como proposta para a integração por reduzir a dimensão e manter desempenho semelhante, com uma pequena melhora observada.

StandardScaler e RobustScaler apresentaram F1-macro de 0,8704 e 0,8706, respectivamente, sem vantagem relevante sobre a referência. Isso é esperado para árvores, que costumam ser pouco sensíveis à escala. Portanto, não há evidência de benefício dessas escalas para o modelo usado nesta comparação, mas o resultado não deve ser generalizado para modelos sensíveis à escala. O One-Hot foi escolhido por representar categorias sem criar uma ordem artificial. A demonstração com dados fictícios confirmou o tratamento das categorias desconhecidas nas duas alternativas.

Os experimentos mantiveram as escolhas de tratamento do Membro 1, as mesmas cinco dobras e o mesmo modelo de referência, sem balanceamento. Imputação, codificação e escala foram ajustadas apenas no treino de cada dobra, mantendo a validação intacta. Os resultados por dobra estão em `por_dobra_m3`, o resumo em `tabela_m3` e a função reutilizável em `montar_preprocessamento_m3`. A combinação com balanceamento e outros modelos ainda precisa ser conferida na integração.

## Membro 4 — Redução de dimensionalidade ou seleção de instâncias

Compara a árvore de decisão com e sem PCA, usando 25 ou 50 componentes nas mesmas cinco dobras. Em cada dobra, o PCA é ajustado apenas nos dados de treino, e são comparados o F1, o tempo e o número de atributos.


In [14]:
# Membro 4 — PCA ajustado somente dentro do treino de cada dobra.
from sklearn.decomposition import PCA

experimentos_m4 = {
    "Sem redução": None,
    "PCA — 25 componentes": 25,
    "PCA — 50 componentes": 50,
}

resultados_m4 = []
for nome, n_componentes in experimentos_m4.items():
    print(f"Executando: {nome}")

    for dobra, (itr, iva) in enumerate(INDICES_DOBRAS, start=1):
        X_tr, y_tr = filtrar_linhas_treino(
            X_treino.iloc[itr], y_treino.iloc[itr], CONFIG_M1["linhas_treino"]
        )
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]

        etapas = [("prep", montar_preprocessamento_m3(
            CONFIG_M1, codificacao="onehot", escala="standard"
        ))]
        if n_componentes is not None:
            etapas.append(("pca", PCA(
                n_components=n_componentes, svd_solver="randomized", random_state=SEMENTE
            )))
        etapas.append(("modelo", clone(MODELO_REFERENCIA)))
        pipeline_m4 = Pipeline(etapas)

        inicio = time.perf_counter()
        pipeline_m4.fit(X_tr, y_tr)
        previsoes = pipeline_m4.predict(X_va)
        segundos = time.perf_counter() - inicio

        f1_normal, f1_anomaly = f1_score(
            y_va, previsoes, labels=[0, 1], average=None, zero_division=0
        )
        n_atributos_entrada = pipeline_m4.named_steps["prep"].transform(X_va.head(1)).shape[1]
        if n_componentes is None:
            n_atributos_saida = n_atributos_entrada
            variancia_pca = np.nan
        else:
            n_atributos_saida = n_componentes
            variancia_pca = pipeline_m4.named_steps["pca"].explained_variance_ratio_.sum()

        resultados_m4.append({
            "experimento": nome,
            "dobra": dobra,
            "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal,
            "f1_anomaly": f1_anomaly,
            "atributos_entrada": n_atributos_entrada,
            "atributos_saida": n_atributos_saida,
            "reducao_pct": 100 * (1 - n_atributos_saida / n_atributos_entrada),
            "variancia_pca": variancia_pca,
            "segundos": segundos,
        })

por_dobra_m4 = pd.DataFrame(resultados_m4)
referencia_m4 = por_dobra_m4[
    por_dobra_m4["experimento"] == "Sem redução"
].set_index("dobra")["f1_macro"]
por_dobra_m4["delta_vs_referencia"] = (
    por_dobra_m4["f1_macro"] - por_dobra_m4["dobra"].map(referencia_m4)
)

tabela_m4 = por_dobra_m4.groupby("experimento", sort=False).agg(
    f1_macro_media=("f1_macro", "mean"),
    f1_macro_dp=("f1_macro", "std"),
    f1_normal=("f1_normal", "mean"),
    f1_anomaly=("f1_anomaly", "mean"),
    delta_vs_referencia=("delta_vs_referencia", "mean"),
    atributos_entrada=("atributos_entrada", "mean"),
    atributos_saida=("atributos_saida", "mean"),
    reducao_pct=("reducao_pct", "mean"),
    variancia_pca=("variancia_pca", "mean"),
    segundos_media=("segundos", "mean"),
)

with pd.option_context("display.float_format", "{:.4f}".format):
    display(tabela_m4)
    print("F1-macro por dobra:")
    display(por_dobra_m4.pivot(index="experimento", columns="dobra", values="f1_macro"))

Executando: Sem redução
Executando: PCA — 25 componentes
Executando: PCA — 50 componentes


,f1_macro_media,f1_macro_dp,f1_normal,f1_anomaly,delta_vs_referencia,atributos_entrada,atributos_saida,reducao_pct,variancia_pca,segundos_media
experimento,,,,,,,,,,
Sem redução,0.8704,0.0045,0.9845,0.7562,0.0000,117.0000,117.0000,0.0000,NaN,1.3316
PCA — 25 componentes,0.8397,0.0056,0.9813,0.6980,-0.0307,117.0000,25.0000,78.6325,0.9067,3.8244
PCA — 50 componentes,0.8537,0.0047,0.9827,0.7246,-0.0167,117.0000,50.0000,57.2650,0.9987,10.3389


F1-macro por dobra:


dobra,1,2,3,4,5
experimento,,,,,
PCA — 25 componentes,0.8345,0.8465,0.8448,0.8352,0.8373
PCA — 50 componentes,0.8534,0.8568,0.8558,0.8456,0.8566
Sem redução,0.8682,0.8697,0.8779,0.8659,0.8701


### Resultados e justificativa — Membro 4

A comparação foi realizada com as mesmas cinco dobras, utilizando a Árvore de Decisão como modelo de referência, sem balanceamento. Todas as etapas de pré-processamento foram ajustadas apenas com os dados de treino de cada dobra.

| Estratégia | F1-macro médio ± DP | F1 normal | F1 anomaly | Dimensões | Redução | Variância PCA | Tempo médio por dobra |
|---|---:|---:|---:|---:|---:|---:|---:|
| Sem redução | 0,8704 ± 0,0045 | 0,9845 | 0,7562 | 117 → 117 | 0,0% | — | 1,46 s |
| PCA (25 componentes) | 0,8397 ± 0,0056 | 0,9813 | 0,6980 | 117 → 25 | 78,6% | 90,67% | 3,96 s |
| PCA (50 componentes) | 0,8537 ± 0,0047 | 0,9827 | 0,7246 | 117 → 50 | 57,3% | 99,87% | 9,39 s |

Os resultados mostram que o PCA reduziu o número de dimensões, mas também diminuiu o desempenho do modelo. Mesmo preservando 99,87% da variância com 50 componentes, o F1 da classe anomaly caiu de 0,7562 para 0,7246. Além disso, o uso de PCA aumentou o tempo de execução.

**Conclusão:** para este modelo e pré-processamento, o PCA não trouxe melhorias. Por isso, a configuração sem redução de dimensões foi a melhor escolha. Essa conclusão vale para este experimento e não significa que o PCA seja inadequado para outros modelos. A decisão foi baseada apenas na validação interna, sem utilizar o conjunto de teste oficial.

## Membro 5 — Integração e comparação das estratégias

**Tarefa:** Combinar os blocos dos membros 1–4 em pipeline sem vazamento. Definir dobras internas comuns e modelo de referência com a equipe; executar comparações explícitas, preferencialmente alterando um fator por vez. Auditar duplicatas entre dobras e consolidar os resultados.

**Entregável:** Tabela estratégia × modelo de referência × métricas, resultados por dobra, discussão das escolhas e configurações para a etapa 3. Não apresentar só uma combinação final.


In [15]:
# TODO — Membro 5: Integração e comparação das estratégias
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 5

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Revisão conjunta

Cada membro deve explicar sua contribuição e revisar a integração com os demais. Conferir os requisitos da etapa, resultados reais, ausência de vazamento, reprodução e declaração de uso de IA antes de entregar.
